# Decision Tree Classification on Abalone Dataset

## Step 1

Read abalone.data as a CSV file with no header.

In [1]:
import pandas as pd

In [2]:
## Direct raw data URL from the UCI repository
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"

## Define column names based on the variables table from https://archive.ics.uci.edu/dataset/1/abalone
column_names = [
    "Sex", "Length", "Diameter", "Height", "Whole Weight", "Shucked Weight", "Viscera Weight", "Shell Weight", "Rings"
]

## Read the data file into a DataFrame
df = pd.read_csv(data_url, header=None, names=column_names)
# print(df)

Construct a new dataset with 4 attributes and a class label, as described in the table below, containing all data from the original dataset.

| Attribute  | Data Type          |Note                       |
|:-----------|:-------------------|:--------------------------|
|Length      |numerical           |                           |
|Diameter    |numerical           |                           |
|Height      |numerical           |                           |
|Whole Weight|numerical           |                           |
|Mature      |normal (class label)|1 if Rings > 8; otherwise 0|

In [3]:
## Select the 4 attributes
new_df = df[["Length", "Diameter", "Height", "Whole Weight"]].copy()

## Add the binary class label
new_df["Mature"] = (df["Rings"] > 8).astype(int)

Show a preview of the first 3 rows of this new dataset.

In [4]:
## Show a preview of the first 3 rows
display(new_df.head(3))

,Length,Diameter,Height,Whole Weight,Mature
0,0.455,0.365,0.095,0.5140,1
1,0.350,0.265,0.090,0.2255,0
2,0.530,0.420,0.135,0.6770,1


## Step 2

Split the dataset into training and testing data by the ratio of 8:2.

In [5]:
from sklearn.model_selection import train_test_split

## Separate features (X) and target label (y)
X = new_df[['Length', 'Diameter', 'Height', 'Whole Weight']]
y = new_df['Mature']

## Split dataset into training (80%) and testing (20%) sets
## "random_state": to control the shuffling applied to the data before applying the split
## "y": the target variable (or label) to predict
## stratify=y: to split the data in a way that preserves the same proportion of class labels in both the training and testing sets as found in the original dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Verify the 8:2 ratio
dataset_size = len(new_df)
print(f"Training set size (80%): {len(X_train)/dataset_size}")
print(f"Testing set size (20%): {len(X_test)/dataset_size}")
# print(f"Training set size (80%): {len(y_train)/dataset_size}"")
# print(f"Testing set size (20%): {len(y_test)/dataset_size}"")

Training set size (80%): 0.7998563562365334
Testing set size (20%): 0.2001436437634666


## Step 3

Perform parameter tuning using `GridSearchCV` as follows:
- Use training data only
- Tune for `DecisionTreeClassifier` from `scikit-learn`. Use only the 4 attributes listed above plus the "Mature" attribute as the target.
- Use f1-score as the score
- Tune the following parameters:
  - `criterion`: `gini` or `entropy`
  - `max_depth`: 3 to 10

In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

## Create a decision tree model object
dt = DecisionTreeClassifier(random_state=42)

## param_grid: for GridSearchCV to test and combine hyperparameter values for the decision tree model
## 'criterion':['gini', 'entropy']: to test 2 different mathematical formulas used to measure the quality of a split
## 'max_depth':list(range(3, 11)): to test different maximum depths for the tree (3 to 10) to prevent overfitting
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': list(range(3, 11)),
}

## GridSearchCV: for hyperparameter tuning and model optimization
## estimator=dt: to test this Decision Tree algorithm structure with different hyperparameter combinations to find which configuration performs best
## param_grid=param_grid: for GridSearchCV to test every possible combination of these parameters (2 criteria x 8 depths = 16 model configurations)
## scoring='f1': to see which parameter combination yields the highest F1-score
## cv=5: to use 5-fold cross-validation
## n_jobs=-1: to use all available CPU cores to test the model configurations (2 criteria x 8 depths x 5 folds = 80 total fits) simultaneously
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, scoring='f1', n_jobs=-1, cv=5)

## Use training data to execute the actual hyperparameter tuning process
grid_search.fit(X_train, y_train)

print(f"Best combination of parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1-score: {grid_search.best_score_:.4f}")

Best combination of parameters: {'criterion': 'entropy', 'max_depth': 3}
Best cross-validation F1-score: 0.8746


## Step 4

Build a decision tree using the best parameter obtained in the previous step using the training data. Visualize the resulting decision tree in text form.

In [7]:
from sklearn.tree import export_text

## Retrieve the optimal model trained on X_train during grid search
best_dt = grid_search.best_estimator_

## Export the decision tree structure to text format
## feature_names=: to tell export_text which labels to use when writing out the tree splits
## feature_names=list(X_train.columns): to map the feature names to the nodes of the decision tree text representation
## show_weights=True: to display the number of training samples belonging to each class that reach each node or leaf
## e.g., weights: [373.00, 79.00] class: 0 means 373 samples of class 0 and 79 samples of class 1
tree_rules = export_text(
    best_dt, feature_names=list(X_train.columns), show_weights=True
)

print(tree_rules)

|--- Whole Weight <= 0.65
|   |--- Height <= 0.10
|   |   |--- Diameter <= 0.22
|   |   |   |--- weights: [176.00, 0.00] class: 0
|   |   |--- Diameter >  0.22
|   |   |   |--- weights: [373.00, 79.00] class: 0
|   |--- Height >  0.10
|   |   |--- Height <= 0.12
|   |   |   |--- weights: [238.00, 150.00] class: 0
|   |   |--- Height >  0.12
|   |   |   |--- weights: [121.00, 208.00] class: 1
|--- Whole Weight >  0.65
|   |--- Height <= 0.15
|   |   |--- Whole Weight <= 1.24
|   |   |   |--- weights: [155.00, 533.00] class: 1
|   |   |--- Whole Weight >  1.24
|   |   |   |--- weights: [0.00, 23.00] class: 1
|   |--- Height >  0.15
|   |   |--- Whole Weight <= 1.28
|   |   |   |--- weights: [52.00, 660.00] class: 1
|   |   |--- Whole Weight >  1.28
|   |   |   |--- weights: [10.00, 563.00] class: 1



## Step 5

Evaluate the decision tree using the testing data. Output the accuracy, precision, recall, and F1-score with respect to class Mature=1. The output results must be properly labelled.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

## Predict class labels for the testing dataset
y_pred = best_dt.predict(X_test)

## Calculate performance metrics with respect to class Mature=1
## pos_label=1: to treat class 1 (Mature) as the positive class when calculating these scores
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

## Show evaluation metrics
print("===== Decision Tree Evaluation on Test Set =====")
print(f"Accuracy:\t{accuracy:.4f}")
print(f"Precision (1):\t{precision:.4f}")
print(f"Recall (1):\t{recall:.4f}")
print(f"F1-Score (1):\t{f1:.4f}")
print("=" * 50 + "\n")

## Optional
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Immature (0)", "Mature (1)"]))

===== Decision Tree Evaluation on Test Set =====
Accuracy:	0.8086
Precision (1):	0.8339
Recall (1):	0.8881
F1-Score (1):	0.8601

Classification Report:
              precision    recall  f1-score   support

Immature (0)       0.75      0.65      0.70       282
  Mature (1)       0.83      0.89      0.86       554

    accuracy                           0.81       836
   macro avg       0.79      0.77      0.78       836
weighted avg       0.80      0.81      0.81       836

